# TS1 Box Model

This tutorial walks through a coupled TUV-x / MICM box model simulation using the TS1 (Tropospheric Standard 1) atmospheric chemistry mechanism.

The simulation runs over a vertical column of 9 altitude grid cells (1–10 km). For each grid cell, TUV-x provides photolysis rate constants, `ussa1976` provides temperature and pressure from the US Standard Atmosphere, and MICM integrates the chemical kinetics forward in time.

Before working through this tutorial, it is recommended to complete:
- [Tutorial 1: Multiple Grid Cells](1.%20multiple_grid_cells.ipynb) — covers MICM state setup and multi-cell solving
- [Tutorial 8: TUV-x Standard Configurations](8.%20tuv-x_standard_configurations.ipynb) — covers computing photolysis rate constants with TUV-x

## 1. Imports

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import ussa1976
import xarray as xr

import musica
from musica.mechanism_configuration import Parser
from musica.micm.solver_result import SolverState
from musica.tuvx import vTS1
from musica.utils import find_config_path

## 2. Compute Photolysis Rate Constants with TUV-x

The vTS1 configuration bundles TUV-x with the TS1/TSMLT photolysis reactions. We run it at solar zenith angle 0° (sun directly overhead) and an Earth–Sun distance of 1 AU to get photolysis rate constants at each altitude grid cell.

The TS1 config file also contains an `aliasing` table that maps TUV-x reaction labels to the `USER.j*` parameter names expected by the MICM mechanism. We load those mappings so we can pass the photolysis rates to the solver.

In [ ]:
tuvx = vTS1.get_tuvx_calculator()
tuv_rates = tuvx.run(sza=0.0, earth_sun_distance=1.0)

tuv_path = find_config_path("tuvx", "ts1_tsmlt.json")
with open(tuv_path, 'r') as f:
    data = json.load(f)
alias_mappings = data.get('__CAM options', {}).get('aliasing', {}).get('pairs', {})

print(f"TUV-x returned rates for {len(tuv_rates.reaction)} reactions")
print(f"Altitude grid edges: {tuv_rates.vertical_edge.values} km")

The TUV-x altitude grid starts at 0 km. We skip the 0 km level (index 0) because it represents the surface boundary and is not a meaningful well-mixed box. We use grid cells at indices 1–9, which correspond to 1–10 km.

In [ ]:
# index 0 is 0 km (surface), skip it; index 10 corresponds to 10 km
start = 1
end = 10

photolysis_rate_constants = {}
for mapping in alias_mappings:
    label = mapping['to']
    scale = mapping.get("scale by", 1)
    tuv_label = mapping['from']
    rate = tuv_rates.sel(reaction=tuv_label).photolysis_rate_constants.values * scale
    photolysis_rate_constants[f'USER.{label}'] = rate[start:end]

print(f"Photolysis rates prepared for {len(photolysis_rate_constants)} reactions across {end - start} grid cells")

## 3. Load the Mechanism and Create the Solver

We parse the TS1 mechanism from its JSON configuration file and create a MICM solver using the Rosenbrock standard-order integrator. Then we create a solver state for the 9 grid cells.

In [ ]:
parser = Parser()
mechanism = parser.parse(find_config_path("v1", "ts1", "ts1.json"))

solver = musica.MICM(
    mechanism=mechanism,
    solver_type=musica.SolverType.rosenbrock_standard_order
)

num_grid_cells = tuv_rates.vertical_edge[start:end].size
state = solver.create_state(num_grid_cells)

print(f"Solver created with {num_grid_cells} grid cells")
print(f"Mechanism has {len(list(state.get_species_ordering()))} species")

## 4. Load Initial Conditions

The initial conditions CSV contains parameters for all grid cells. Each row has a parameter name and one or two values, following this convention:

| Prefix | value1 | value2 |
|--------|--------|--------|
| `CONC` | Initial concentration (mol m⁻³) | — |
| `ENV`  | Temperature (K) or Pressure (Pa) | — |
| `USER` | User-defined parameter value | — |
| `SURF` | Effective radius (m) | Particle number concentration (# m⁻³) |

We split the file into these four groups and process each separately.

In [ ]:
conditions = pd.read_csv(
    find_config_path("v1", "ts1", "initial_conditions.csv"),
    sep=',',
    names=['parameter', 'value1', 'value2'],
    dtype={'parameter': str, 'value1': float, 'value2': float}
)

surface_reactions       = conditions[conditions['parameter'].str.contains('SURF')]
initial_concentrations  = conditions[conditions['parameter'].str.contains('CONC')].copy()
environmental_conditions = conditions[conditions['parameter'].str.contains('ENV')]
user_defined_conditions = conditions[conditions['parameter'].str.contains('USER')]

# Strip the 'CONC.' prefix so names match the species ordering expected by the solver
initial_concentrations.loc[:, 'parameter'] = (
    initial_concentrations['parameter'].str.replace('CONC.', '', regex=False)
)

assert (
    len(surface_reactions) + len(initial_concentrations) +
    len(environmental_conditions) + len(user_defined_conditions)
) == len(conditions)

print(f"Loaded {len(initial_concentrations)} species, {len(user_defined_conditions)} user-defined params, "
      f"{len(surface_reactions)} surface reactions")

## 5. Set State Conditions

We populate the solver state with:
- **Concentrations** from the initial conditions CSV, broadcast to all grid cells.
- **Temperature and pressure** from the US Standard Atmosphere (`ussa1976`), evaluated at the TUV-x altitude grid edges for our 9 cells.
- **User-defined rate parameters** from the CSV, overwritten with the TUV-x photolysis rates where applicable.
- **Surface reaction parameters** (effective radius and particle number concentration) also from the CSV.

In [ ]:
# Build concentration dict — same value broadcast to every grid cell
concentration_dict = {
    row['parameter']: [row['value1']] * num_grid_cells
    for _, row in initial_concentrations.iterrows()
}

# Build user-defined rate parameters dict
user_defined_dict = {
    row['parameter']: [row['value1']] * num_grid_cells
    for _, row in user_defined_conditions.iterrows()
}

# Surface reactions require two parameters: effective radius and particle number concentration
for _, row in surface_reactions.iterrows():
    user_defined_dict[f"{row['parameter']}.effective radius [m]"] = [row['value1']] * num_grid_cells
    user_defined_dict[f"{row['parameter']}.particle number concentration [# m-3]"] = [row['value2']] * num_grid_cells

# US Standard Atmosphere temperature and pressure at the TUV-x grid edges (convert km -> m)
atm = ussa1976.compute(z=tuv_rates.vertical_edge[start:end].data * 1000, variables=["t", "p"])
temperature = atm['t'].values
pressure    = atm['p'].values

# Overwrite scalar USER.j* values with the per-altitude TUV-x photolysis rates
found_rates  = sorted(photolysis_rate_constants.keys())
needed_rates = sorted([k for k in state.get_user_defined_rate_parameters() if 'USER.j' in k])
missing_rates = set(needed_rates) - set(found_rates)
if missing_rates:
    print(f"Missing photolysis rates (will remain at CSV values): {missing_rates}")
user_defined_dict.update(photolysis_rate_constants)

# Apply everything to the state
state.set_conditions(temperature, pressure)
state.set_concentrations(concentration_dict)
state.set_user_defined_rate_parameters(user_defined_dict)

print("State initialised.")
print(f"Temperature range: {temperature.min():.1f} – {temperature.max():.1f} K")
print(f"Pressure range:    {pressure.min():.0f} – {pressure.max():.0f} Pa")

## 6. Run the Simulation

We integrate the chemistry forward using 30-second time steps for 1/10 of a day (~2.4 hours). At each step we call `solver.solve()` in an inner loop that accumulates time until a full time step is completed — this accounts for the solver occasionally taking shorter sub-steps to maintain accuracy.

In [ ]:
times = [0]
concentrations = [state.get_concentrations()]

time_step       = 30                      # seconds per output step
simulation_length = 0.1 * 24 * 60 * 60   # 1/10 of a day in seconds
current_time    = 0
last_printed_percent = -5

while current_time < simulation_length:
    elapsed = 0
    while elapsed < time_step:
        remaining = time_step - elapsed
        result = solver.solve(state, remaining)
        elapsed      += result.stats.final_time
        current_time += result.stats.final_time
        if result.state != SolverState.Converged:
            print(f"  Solver state: {result.state} at t={current_time:.1f} s")

    current_percent = (current_time / simulation_length) * 100
    if int(current_percent // 5) * 5 > last_printed_percent:
        last_printed_percent = int(current_percent // 5) * 5
        print(f"Progress: {last_printed_percent}%")

    times.append(current_time)
    concentrations.append(state.get_concentrations())

print("Simulation complete.")

## 7. Collect Results into an xarray Dataset

We organise the simulation output into an `xr.Dataset` with dimensions `time` and `grid_cell`. Species concentrations vary over both dimensions; environmental conditions and user-defined rate parameters vary only over `grid_cell` (they are held constant in this simulation).

In [ ]:
final_conditions = state.get_conditions()

data_vars = {
    "temperature": (["grid_cell"], final_conditions["temperature"]),
    "pressure":    (["grid_cell"], final_conditions["pressure"]),
    "air density": (["grid_cell"], final_conditions["air_density"]),
}

# User-defined rate parameters (stored as a 2D array: parameter × grid_cell)
user_params = state.get_user_defined_rate_parameters()
user_param_names  = list(user_params.keys())
user_param_values = [user_params[name] for name in user_param_names]
if user_param_names:
    data_vars["user_defined_rate_parameters"] = (["user_parameter", "grid_cell"], user_param_values)

# Species concentrations (time × grid_cell)
species_ordering = state.get_species_ordering()
for species in species_ordering:
    data_vars[species] = (
        ["time", "grid_cell"],
        [concentrations[t][species] for t in range(len(concentrations))]
    )

coords = {
    "time":      times,
    "grid_cell": range(num_grid_cells),
}
if user_param_names:
    coords["user_parameter"] = user_param_names

ds = xr.Dataset(data_vars, coords=coords)
print(ds)

## 8. Plot Results

We plot the time evolution of four representative species — BEPOMUC, C6H5OOH, BR, and CL — for every grid cell. Each line corresponds to one altitude.

In [ ]:
time_hours = ds['time'] / 3600  # convert seconds to hours
species_to_plot = ['BEPOMUC', 'C6H5OOH', 'BR', 'CL']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, species in zip(axes.flat, species_to_plot):
    for cell in range(num_grid_cells):
        ax.plot(time_hours, ds[species].sel(grid_cell=cell), label=f"{cell + start} km")
    ax.set_title(species)
    ax.set_xlabel('Time [hours]')
    ax.set_ylabel('Concentration [mol m⁻³]')
    ax.set_xlim(0, simulation_length / 3600)
    ax.set_ylim(0, None)
    ax.grid(True, alpha=0.5)
    ax.spines[:].set_visible(False)
    ax.tick_params(width=0)

# Add a shared legend outside the last axis
handles, labels = axes.flat[-1].get_legend_handles_labels()
fig.legend(handles, labels, title='Altitude', loc='lower center', ncol=num_grid_cells, bbox_to_anchor=(0.5, -0.05))

fig.tight_layout()
plt.show()

## 9. Save Output (Optional)

The dataset can be saved to a NetCDF file for later analysis.

In [ ]:
ds.to_netcdf('ts1_box_model.nc')
print("Saved to ts1_box_model.nc")